In [1]:
!pip install xgboost==2.1.4 -q

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

PATH = "/content/drive/MyDrive/fraud-detection"
X_train, X_test, y_train, y_test = joblib.load(f"{PATH}/data/split.pkl")

results = []

def evaluate(name, model):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    r = {
        "model": name,
        "precision": round(precision_score(y_test, pred), 4),
        "recall": round(recall_score(y_test, pred), 4),
        "f1": round(f1_score(y_test, pred), 4),
        "pr_auc": round(average_precision_score(y_test, proba), 4),
    }
    results.append(r)
    print(r)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000))
])
lr.fit(X_train, y_train)
evaluate("LogReg + class_weight", lr)

{'model': 'LogReg + class_weight', 'precision': 0.0435, 'recall': 0.8996, 'f1': 0.0831, 'pr_auc': np.float64(0.5751)}


In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=15,
                            class_weight="balanced", n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
evaluate("RandomForest + class_weight", rf)

{'model': 'RandomForest + class_weight', 'precision': 1.0, 'recall': 0.9963, 'f1': 0.9982, 'pr_auc': np.float64(0.9979)}


In [5]:
from xgboost import XGBClassifier

ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                    scale_pos_weight=ratio, eval_metric="aucpr",
                    n_jobs=-1, random_state=42)
xgb.fit(X_train, y_train)
evaluate("XGBoost + scale_pos_weight", xgb)

{'model': 'XGBoost + scale_pos_weight', 'precision': 0.9025, 'recall': 0.9921, 'f1': 0.9452, 'pr_auc': np.float64(0.9934)}


In [6]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

def make_xgb():
    return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                         eval_metric="aucpr", n_jobs=-1, random_state=42)

xgb_smote = ImbPipeline([("smote", SMOTE(sampling_strategy=0.1, random_state=42)),
                         ("clf", make_xgb())])
xgb_smote.fit(X_train, y_train)
evaluate("XGBoost + SMOTE", xgb_smote)

xgb_under = ImbPipeline([("under", RandomUnderSampler(sampling_strategy=0.1, random_state=42)),
                         ("clf", make_xgb())])
xgb_under.fit(X_train, y_train)
evaluate("XGBoost + Undersampling", xgb_under)

{'model': 'XGBoost + SMOTE', 'precision': 0.9032, 'recall': 0.9878, 'f1': 0.9436, 'pr_auc': np.float64(0.9934)}
{'model': 'XGBoost + Undersampling', 'precision': 0.8539, 'recall': 0.9963, 'f1': 0.9197, 'pr_auc': np.float64(0.9908)}


In [7]:
res = pd.DataFrame(results).sort_values("pr_auc", ascending=False)
display(res)
res.to_csv(f"{PATH}/figures/results.csv", index=False)
joblib.dump(xgb, f"{PATH}/models/xgb.pkl")

,model,precision,recall,f1,pr_auc
1,RandomForest + class_weight,1.0000,0.9963,0.9982,0.9979
3,XGBoost + SMOTE,0.9032,0.9878,0.9436,0.9934
2,XGBoost + scale_pos_weight,0.9025,0.9921,0.9452,0.9934
4,XGBoost + Undersampling,0.8539,0.9963,0.9197,0.9908
0,LogReg + class_weight,0.0435,0.8996,0.0831,0.5751


['/content/drive/MyDrive/fraud-detection/models/xgb.pkl']

In [8]:
cols = ["errorBalanceOrig", "errorBalanceDest"]
xgb2 = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                     scale_pos_weight=ratio, eval_metric="aucpr", n_jobs=-1, random_state=42)
xgb2.fit(X_train.drop(columns=cols), y_train)
proba = xgb2.predict_proba(X_test.drop(columns=cols))[:, 1]
print("PR-AUC without errorBalance:", average_precision_score(y_test, proba))

PR-AUC without errorBalance: 0.9602707144682863
